In [ ]:
from pathlib import Path
import json 
import pandas as pd
import re
import networkx as nx
import matplotlib.pyplot as plt
import raphtory as rp
import raphtory.algorithms as rpa

In [ ]:
# relative path to load LLM traces

data_path = Path("data") / "MAD_full_dataset.json"

MAS_data = pd.read_json(data_path)
print(MAS_data.head())

EXTRACTING NON FAILING LOGS

In [ ]:
failure_mode_2_cols = ['2.1', '2.2', '2.3', '2.4', '2.5', '2.6']
clean_logs_mask = MAS_data['mast_annotation'].apply(lambda tation: all(tation[k] == 0 for k in failure_mode_2_cols))
clean_logs = MAS_data[clean_logs_mask]['trace_id'].tolist()
print(f'Traces with no intra agent misalignment failure: {len(clean_logs)}')

In [ ]:
MAS_data[clean_logs_mask].groupby('mas_name')['trace_id'].count()

EXTRACTING FAILING LOGS

In [ ]:
target_failure_mode_cols = ['2.1', '2.2', '2.3', '2.4', '2.5', '2.6']
target_failed_logs_mask = MAS_data['mast_annotation'].apply(lambda tation: any(tation[k] == 1 for k in target_failure_mode_cols))
target_failed_logs = MAS_data[target_failed_logs_mask]['trace_id'].tolist()
print(f'Traces with target failures: {len(target_failed_logs)}')
MAS_data[target_failed_logs_mask].groupby('mas_name')['trace_id'].count()

1. Hyperagent failures

In [ ]:
matplotlib_fail = MAS_data[target_failed_logs_mask & (MAS_data['mas_name'] == 'HyperAgent') & (MAS_data['trace_id'] == 5)].reset_index(drop=True)

In [ ]:
plt_fail_traj = matplotlib_fail.iloc[0]['trace']['trajectory']
print(plt_fail_traj)

In [ ]:
sympy_fail = MAS_data[target_failed_logs_mask & (MAS_data['mas_name'] == 'HyperAgent') & (MAS_data['trace_id'] == 10)].reset_index(drop=True)

In [ ]:
#checking trajecory matches raw log 
sympy_fail_traj = sympy_fail.iloc[0]['trace']['trajectory']
print(sympy_fail_traj)

2. Hyperagent success

In [ ]:
matplotlib_success = MAS_data[clean_logs_mask & (MAS_data['mas_name'] == 'HyperAgent') & (MAS_data['trace_id'] == 1)].reset_index(drop=True)

In [ ]:
#checking trajectory matches raw log 
plt_success_traj = matplotlib_success.iloc[0]['trace']['trajectory']
print(plt_success_traj)

In [ ]:
sympy_success = MAS_data[clean_logs_mask & (MAS_data['mas_name'] == 'HyperAgent') & (MAS_data['trace_id'] == 12)].reset_index(drop=True)

In [ ]:
#checking trajectory matches raw log
sympy_success_traj = sympy_success.iloc[0]['trace']['trajectory']
print(sympy_success_traj)

**STARTING TO BUILD GRAPHS**

In [ ]:
#extracting role names for nodes based off metadata structures identified during log analysis (see report Methods section for detail)
def extract_roles(trajectory):
    pattern = r"INFO - (.+?)(?:\'s Response|->)"
    return re.findall(pattern, trajectory)

In [ ]:
#checking role names align to mmanual log exploration - they do 
roles_plt_fail = set(extract_roles(plt_fail_traj))
print(roles_plt_fail)

In [ ]:
#investigating messages sent and recieved between roles to check they match log analysis - they do 

def planner_messages(trajectory):
    sent_pattern = r'Intern Name: (\w+ \w+)'
    received_pattern = r'(Navigator|Editor|Executor)->Planner'
    sent = re.findall(sent_pattern, trajectory)
    received = re.findall(received_pattern, trajectory)
    print('Planner sends to:', sent)
    print('Planner recieved from:', received)    

In [ ]:
planner_messages(plt_fail_traj)

In [ ]:
#normalising roles as role and inner-role-assistant are the same e.g. navigator & inner-navigator-assistant are the same role
#see HyperAgent framework overview (Table 1) in report for details
def normalise_role(role):
    role = role.strip()
    role = role.replace('Inner-', '').replace('-Assistant', '')
    role = role.replace('Codebase ', '')
    return role

test_names = ['Inner-Navigator-Assistant', 'Inner-Editor-Assistant', 
              'Inner-Executor-Assistant', 'Codebase Editor', 
              'Codebase Navigator', 'Planner', 'Navigator', 
              'Editor', 'Executor']

for name in test_names:
    print(f"{name} -> {normalise_role(name)}")

In [ ]:
#extracting messages exchanged to get communication structure network edges
def extract_message_edge(trajectory):
    edges = []
    
    # Planner -> Specialist (Planner dispatches)
    dispatches = re.findall(r'Intern Name: (.+)', trajectory)
    for target in dispatches:
        edges.append(('Planner', normalise_role(target)))
    
    # Specialist -> Planner (Specialist reports back)
    reports = re.findall(r'(Navigator|Editor|Executor)->Planner', trajectory)
    for sender in reports:
        edges.append((normalise_role(sender), 'Planner'))
    
    return edges

# Test on matplotlib failed trace
edges = extract_message_edge(plt_fail_traj)
for e in edges:
    print(e)

In [ ]:
#extract edges (message sent) from each trace 
plt_fail_edges = extract_message_edge(plt_fail_traj)
plt_success_edges = extract_message_edge(plt_success_traj)
sympy_fail_edges = extract_message_edge(sympy_fail_traj)
sympy_success_edges = extract_message_edge(sympy_success_traj)

#build static directed graph for each trace
G_plt_fail_static = nx.DiGraph()
G_plt_success_static = nx.DiGraph()
G_sympy_fail_static = nx.DiGraph()
G_sympy_success_static = nx.DiGraph()

for G, edges in [
    (G_plt_fail_static, plt_fail_edges),
    (G_plt_success_static, plt_success_edges),
    (G_sympy_fail_static, sympy_fail_edges),
    (G_sympy_success_static, sympy_success_edges)
]:
    for sender, receiver in edges:
        if G.has_edge(sender, receiver):
            G[sender][receiver]['weight'] += 1
        else:
            G.add_edge(sender, receiver, weight=1)

#print network structures 
for name, G in [('Matplotlib fail', G_plt_fail_static), 
                ('Matplotlib success', G_plt_success_static),
                ('Sympy fail', G_sympy_fail_static), 
                ('Sympy success', G_sympy_success_static)]:
    print(f"{name} | Nodes: {list(G.nodes())} | Edges: {G.number_of_edges()}")

**METRICS CALCULATED FOR REPORT - STATIC, UN-WEIGHTED COMMUNICATION STRUCTURE**

1. Density 
2. Degree & degree centrality

In [ ]:
#density 
print(f'{nx.density(G_plt_fail_static)}')
print(f'{nx.density(G_plt_success_static)}')
print(f'{nx.density(G_sympy_fail_static)}')
print(f'{nx.density(G_plt_success_static)}')

In [ ]:
#plt trace
print(f'Plt fail in degree: {G_plt_fail_static.in_degree()}')
print(f'Plt fail out degree: {G_plt_fail_static.out_degree()}')
print(f'Plt success in degree: {G_plt_success_static.in_degree()}')
print(f'Plt success out degree: {G_plt_success_static.out_degree()}')

print(f'Plt fail in degree centrality: {nx.out_degree_centrality(G_plt_fail_static)}')
print(f'Plt fail out degree centrality: {nx.in_degree_centrality(G_plt_fail_static)}')

print(f'Plt success in degree centrality: {nx.in_degree_centrality(G_plt_fail_static)}')
print(f'Plt success out degree centrality: {nx.out_degree_centrality(G_plt_fail_static)}')

#sympy trace
print(f'Sympy fail in degree: {G_sympy_fail_static.in_degree()}')
print(f'Sympy fail out degree: {G_sympy_fail_static.out_degree()}')
print(f'Sympy success in degree: {G_sympy_success_static.in_degree()}')
print(f'Sympy success out degree: {G_sympy_success_static.out_degree()}')

print(f'Sympy fail in degree centrality: {nx.out_degree_centrality(G_sympy_fail_static)}')
print(f'Sympy fail out degree centrality: {nx.in_degree_centrality(G_sympy_fail_static)}')

print(f'Sympy success in degree centrality: {nx.in_degree_centrality(G_sympy_success_static)}')
print(f'Sympy success out degree centrality: {nx.out_degree_centrality(G_sympy_success_static)}')

#consistent across networks. Offers minimal information beyond planner as hub. Insight comes with weighted & dynamic networks.


In [ ]:
# visualising static directed communication structure graph 
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

#graphs to visualise with titles and associated colour
graphs = [
    (G_plt_fail_static, 'Matplotlib Fail', 'red'),
    (G_plt_success_static, 'Matplotlib Success', 'green'),
    (G_sympy_fail_static, 'Sympy Fail', 'red'),
    (G_sympy_success_static, 'Sympy Success', 'green')
]

#layout
pos = nx.spring_layout(G_plt_fail_static, seed=15)

#building each plot
for ax, (G, title, colour) in zip(axes, graphs):
    nx.draw(G, pos, ax=ax,
            with_labels=True,
            node_color='lightblue',
            node_size=2000,
            edge_color=colour,
            width=2,
            arrows=True,
            arrowsize=20,
            connectionstyle='arc3,rad=0.2')
    
    ax.set_title(title, fontweight='bold', color=colour)
    ax.axis('off')

plt.suptitle('Network 1a: Static Communication Structure. Node: agent, Edge: message exchange', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('network1_a_.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

**BUILDING WEIGHTED DIRECTED NETWORK**

In [ ]:
#calculating graph weights
for name, G in [('Matplotlib fail', G_plt_fail_static), 
                ('Matplotlib success', G_plt_success_static),
                ('Sympy fail', G_sympy_fail_static), 
                ('Sympy success', G_sympy_success_static)]:
    print(f"\n{name}:")
    for u, v, d in G.edges(data=True):
        print(f"  {u} -> {v}: weight={d['weight']}")

**Weighted network metrics**
1. weighted in degree & out degree
2. PageRank 

In [ ]:
#plt trace: weighted network 
print(f"Plt fail weighted in degree: {G_plt_fail_static.in_degree(weight='weight')}")
print(f"Plt fail weighted out degree: {G_plt_fail_static.out_degree(weight='weight')}")
print(f"Plt success weighted in degree: {G_plt_success_static.in_degree(weight='weight')}")
print(f"Plt success weighted out degree: {G_plt_success_static.out_degree(weight='weight')}")

#sympy trace: weighted network 
print(f"Sympy fail weighted in degree: {G_sympy_fail_static.in_degree(weight='weight')}")
print(f"Sympy fail weighted out degree: {G_sympy_fail_static.out_degree(weight='weight')}")
print(f"Sympy success weighted in degree: {G_sympy_success_static.in_degree(weight='weight')}")
print(f"Sympy success weighted out degree: {G_sympy_success_static.out_degree(weight='weight')}")

In [ ]:
import scipy

In [ ]:
#page rank algorithm to analyse the relative importance of each node in terms of messages exchanged with other nodes
print(f"Plt fail page rank: {nx.pagerank(G_plt_fail_static, weight='weight')}")
print(f"Plt success page rank: {nx.pagerank(G_plt_success_static, weight='weight')}")
print(f"Sympy fail page rank: {nx.pagerank(G_sympy_fail_static, weight='weight')}")
print(f"Sympy success page rank: {nx.pagerank(G_sympy_success_static, weight='weight')}")


**WEIGHTED STATIC COMMS GAPH**

In [ ]:
#visualising a weighted graph for the static communication network 
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

graphs = [
    (G_plt_fail_static, 'Matplotlib Fail', 'red'),
    (G_plt_success_static, 'Matplotlib Success', 'green'),
    (G_sympy_fail_static, 'Sympy Fail', 'red'),
    (G_sympy_success_static, 'Sympy Success', 'green')
]

#calculate node sizes based on total degree
node_sizes = {node: G.degree(node) * 500 for node in G.nodes()}
sizes = [node_sizes[node] for node in G.nodes()]

#edge width by weight 
global_max = max(
    max(d['weight'] for u, v, d in G.edges(data=True))
    for G in [G_plt_fail_static, G_plt_success_static, G_sympy_fail_static, G_sympy_success_static]
)
#layout
pos = nx.spring_layout(G_plt_fail_static, seed=15)

for ax, (G, title, colour) in zip(axes, graphs):
    edge_widths = [1 + 4 * (G[u][v]['weight'] / global_max) for u, v in G.edges()]
    
    nx.draw(G, pos, ax=ax, with_labels=True,
            node_color='lightblue', 
            node_size=sizes,
            arrows=True, 
            arrowsize=20,
            edge_color=colour, 
            width=edge_widths,
            connectionstyle='arc3,rad=0.2')
    
    #separate labels for each direction
    for (u, v, d) in G.edges(data=True):
        x = (pos[u][0] + pos[v][0]) / 2
        y = (pos[u][1] + pos[v][1]) / 2
        
        # Offset label based on direction
        if u == 'Planner':
            offset = 0.1
        else:
            offset = -0.1
            
        ax.text(x + offset, y + offset, str(d['weight']),
                fontsize=9, fontweight='bold',
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.2',
                         facecolor='white', edgecolor='none', alpha=0.8))
    
    ax.set_title(title)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.suptitle('Network 1b: Weighted Static Communication Structure. Node: agent, Edge: weighted message exchange', fontsize=10, fontweight='bold')
plt.savefig('network1_b_.png', dpi=300, bbox_inches='tight',
            facecolor='white')
plt.show()

2. DYNAMIC NETWORK ANALYSIS

1. extract transitions as time steps

In [ ]:
#extracting transition indicators identified through exploring MAS logs  
def extract_transitions(trajectory):
    steps = re.split(r'HyperAgent_\w+__\w+-\d+', trajectory)
    sequence = []
    for step in steps:
        if "Planner's Response" in step and len(step) > 50:
            sequence.append('Planner')
        elif "Inner-Navigator-Assistant's Response" in step and len(step) > 50:
            sequence.append('Navigator')
        elif "Inner-Editor-Assistant's Response" in step and len(step) > 50:
            sequence.append('Editor')
        elif "Inner-Executor-Assistant's Response" in step and len(step) > 50:
            sequence.append('Executor')
        elif "Navigator->Planner" in step:
            sequence.append('Navigator')
        elif "Editor->Planner" in step:
            sequence.append('Editor')
        elif "Executor->Planner" in step:
            sequence.append('Executor')
    transitions = []
    for i in range(len(sequence) - 1):
        transitions.append((sequence[i], sequence[i+1]))

    return sequence, transitions

# Test on matplotlib fail
seq, transitions = extract_transitions(plt_fail_traj)

print("Sequence:")
for i, s in enumerate(seq):
    print(f"  {i+1}: {s}")

print(f"\nTransitions:")
for t in transitions:
    print(f"  {t[0]} -> {t[1]}")

2. build temporal graphs with raphtory

In [ ]:
def build_raphtory_graph(trajectory):
    g = rp.Graph()
    seq, transitions = extract_transitions(trajectory)
    
    for i, (u, v) in enumerate(transitions):
        g.add_edge(i, u, v)
    
    return g

g_plt_fail_dynamic = build_raphtory_graph(plt_fail_traj)
g_plt_success_dynamic = build_raphtory_graph(plt_success_traj)
g_sympy_fail_dynamic = build_raphtory_graph(sympy_fail_traj)
g_sympy_success_dynamic = build_raphtory_graph(sympy_success_traj)

for name, g in [('Matplotlib fail', g_plt_fail_dynamic),
                ('Matplotlib success', g_plt_success_dynamic),
                ('Sympy fail', g_sympy_fail_dynamic),
                ('Sympy success', g_sympy_success_dynamic)]:
    print(f"{name}: {g}")

**DYNAMIC COMMUNICATION SEQUENCE NETWORK MERICS**
1. Time-respecting connectivity (Nicosia et al, 2012)

In [ ]:
#building truncated graph with time windows for successful graphs to match time windows for failing graphs
def build_raphtory_graph_truncated(trajectory, max_steps=None):
    g = rp.Graph()
    seq, transitions = extract_transitions(trajectory)
    if max_steps:
        transitions = transitions[:max_steps]
    for i, (u, v) in enumerate(transitions):
        g.add_edge(i, u, v)
    return g

g_plt_success_trunc = build_raphtory_graph_truncated(plt_success_traj, max_steps=29)
g_sympy_success_trunc = build_raphtory_graph_truncated(sympy_success_traj, max_steps=32)

In [ ]:
#check connectivity of truncated successful netowrks 
for name, g in [('Matplotlib success truncated', g_plt_success_trunc),
                ('Sympy success truncated', g_sympy_success_trunc)]:
    print(f"\n{name}:")
    print(f"  Strongly connected: {rpa.strongly_connected_components(g)}")
    print(f"  Weakly connected: {rpa.weakly_connected_components(g)}")

In [ ]:
#check connectivity of failing networks
for name, g in [('Matplotlib fail truncated', g_sympy_fail_dynamic),
                ('Sympy fail truncated', g_sympy_fail_dynamic)]:
    print(f"\n{name}:")
    print(f"  Strongly connected: {rpa.strongly_connected_components(g)}")
    print(f"  Weakly connected: {rpa.weakly_connected_components(g)}")

2. Building multi directed graphs for networkx analysis 

In [ ]:
results = {}
for name, traj in [('Matplotlib fail', plt_fail_traj),
                   ('Matplotlib success', plt_success_traj),
                   ('Sympy fail', sympy_fail_traj),
                   ('Sympy success', sympy_success_traj)]:
    
    seq, transitions = extract_transitions(traj)
    G_seq = nx.MultiDiGraph()
    for i, (u, v) in enumerate(transitions):
        G_seq.add_edge(u, v, timestep=i)
    
    results[name] = G_seq

Building temporal networks with pyvis to visualse communication sequences at each time step

In [ ]:
import pyvis
from pyvis.network import Network

In [ ]:
#build and visualise temporal graphs with pyvis
def build_pyvis_temporal(trajectory, title, colour):
    net = Network(height='500px', width='100%', 
                  directed=True, 
                  notebook=True,
                  cdn_resources='remote')
    
    seq, transitions = extract_transitions(trajectory)
    n = len(transitions)
    
    # Add nodes
    for role in ['Planner', 'Navigator', 'Editor', 'Executor']:
        net.add_node(role, label=role, color=colour, size=20)
    
    # Add each transition with opacity based on timestep
    for i, (u, v) in enumerate(transitions):
        # Earlier edges lighter, later edges darker
        opacity = 0.1 + 0.9 * (i / max(n-1, 1))
        
        # Convert opacity to hex alpha
        alpha_hex = format(int(opacity * 255), '02x')
        
        # Append alpha to colour hex
        edge_colour = colour + alpha_hex
        
        net.add_edge(u, v,
                    title=f't={i}',
                    color=edge_colour)
    
    net.toggle_physics(True)
    net.repulsion(
        node_distance=200,
        central_gravity=0.3,
        spring_length=200,
        spring_strength=0.05,
        damping=0.09
    )
    net.show(f'{title}_temporal.html')

build_pyvis_temporal(plt_fail_traj, 'plt_fail', '#ff0000')
build_pyvis_temporal(plt_success_traj, 'plt_success', '#00aa00')
build_pyvis_temporal(sympy_fail_traj, 'sympy_fail', '#ff0000')
build_pyvis_temporal(sympy_success_traj, 'sympy_success', '#00aa00')

In [ ]:
#count transitions within networks (i.e. timesteps)
for name, traj in [('Matplotlib fail', plt_fail_traj),
                   ('Matplotlib success', plt_success_traj),
                   ('Sympy fail', sympy_fail_traj),
                   ('Sympy success', sympy_success_traj)]:
    seq, transitions = extract_transitions(traj)
    print(f"{name}: {len(transitions)} transitions")

In [ ]:
#build and visualise truncated temporal networks where observation window is consistent across both failing and successful networks 
def build_pyvis_temporal_trunc(trajectory, title, colour, max_steps=None):
    net = Network(height='500px', width='100%', 
                  directed=True, 
                  notebook=True,
                  cdn_resources='remote')
    
    seq, transitions = extract_transitions(trajectory)
    
    # Truncate if max_steps specified
    if max_steps:
        transitions = transitions[:max_steps]
    
    n = len(transitions)
    
    # Add nodes
    for role in ['Planner', 'Navigator', 'Editor', 'Executor']:
        net.add_node(role, label=role, color=colour, size=20)
    
    # Add edges
    for i, (u, v) in enumerate(transitions):
        # Earlier edges lighter, later edges darker
        opacity = 0.5 + 0.95 * (i / max(n-1, 1))
        
        # Convert opacity to hex alpha
        alpha_hex = format(int(opacity * 255), '02x')
        
        # Append alpha to colour hex
        edge_colour = colour + alpha_hex
        
        net.add_edge(u, v,
                    title=f't={i}',
                    color=edge_colour)
    
    net.repulsion(node_distance=200, central_gravity=0.3,
                  spring_length=200, spring_strength=0.05, damping=0.09)
    net.show(f'{title}_temporal_truncated.html')

In [ ]:
build_pyvis_temporal_trunc(plt_success_traj, 'plt_success', 'green', max_steps=29)
build_pyvis_temporal_trunc(sympy_success_traj, 'sympy_success', 'green', max_steps=32)


3. TYPE OF COMMUNICATION ACROSS NETWORK ANALYSIS

In [ ]:
#investgating type of communication across network to check alignment with log analysis - it matches up to raw log 
#testing with maplotlib fail  
sample_traj = plt_fail_traj
lines = sample_traj.split('\n')

thought_lines = [(i, l.strip()) for i, l in enumerate(lines) if 'Thought:' in l]
subgoal_lines = [(i, l.strip()) for i, l in enumerate(lines) if 'Subgoal:' in l]
action_lines = [(i, l.strip()) for i, l in enumerate(lines) if 'Action:' in l]

print(f"Thoughts: {len(thought_lines)}")
print(f"Subgoals: {len(subgoal_lines)}")
print(f"Actions: {len(action_lines)}")

print("\nSample thoughts:")
for i, l in thought_lines[:3]:
    print(f"  Line {i}: {l[:100]}")

print("\nSample subgoals:")
for i, l in subgoal_lines[:3]:
    print(f"  Line {i}: {l[:100]}")

print("\nSample actions:")
for i, l in action_lines[:3]:
    print(f"  Line {i}: {l[:100]}")

In [ ]:
#extract details of the type of communication beign shared across MAS at specific time steps 
def extract_communication_type(trajectory):
    steps = re.split(r'HyperAgent_\w+__\w+-\d+', trajectory)
    steps = [s.strip() for s in steps if s.strip()]
    
    communication_type = []
    
    for t, step in enumerate(steps):
        
        # Identify role
        if "Planner's Response" in step:
            role = 'Planner'
        elif "Inner-Navigator-Assistant" in step:
            role = 'Navigator'
        elif "Inner-Editor-Assistant" in step:
            role = 'Editor'
        elif "Inner-Executor-Assistant" in step:
            role = 'Executor'
        elif "Navigator->Planner" in step:
            role = 'Navigator'
        elif "Editor->Planner" in step:
            role = 'Editor'
        elif "Executor->Planner" in step:
            role = 'Executor'
        else:
            continue
        
        # Iterate line by line in order
        lines = step.split('\n')
        for line in lines:
            line = line.strip()
            if 'Subgoal:' in line:
                communication_type.append({'t': t, 'role': role, 'type': 'Subgoal'})
            elif 'Action:' in line and len(line.strip()) < 20:
                communication_type.append({'t': t, 'role': role, 'type': 'Action'})
            elif 'Thought:' in line:
                communication_type.append({'t': t, 'role': role, 'type': 'Thought'})
    
    return communication_type

#test on matplotlib fail - the total communication events matches number extracted in cell
seq = extract_communication_type(plt_fail_traj)
print(f"Total events: {len(seq)}")
for s in seq:
    print(f"  t={s['t']} | {s['role']:10} | {s['type']}")

In [ ]:
#plot the type of communication being shared across MAS at different time steps
def plot_communication_type(trajectory, title, colour, max_steps=None):
    seq = extract_communication_type(trajectory)
    
    if max_steps:
        seq = [s for s in seq if s['t'] <= max_steps]
    
    # Layout
    role_order = {'Planner': 3, 'Navigator': 2, 'Editor': 1, 'Executor': 0}
    type_colours = {
        'Thought': '#3498db',
        'Subgoal': '#e74c3c',
        'Action': '#2ecc71'
    }
    
    fig, ax = plt.subplots(figsize=(20, 6))
    
    # Plot nodes
    for event in seq:
        x = event['t']
        y = role_order[event['role']]
        c = type_colours[event['type']]
        ax.scatter(x, y, color=c, s=200, zorder=3)
    
    # Plot edges between consecutive events
    for i in range(len(seq) - 1):
        x1, y1 = seq[i]['t'], role_order[seq[i]['role']]
        x2, y2 = seq[i+1]['t'], role_order[seq[i+1]['role']]
        ax.annotate('', 
                   xy=(x2, y2), xytext=(x1, y1),
                   arrowprops=dict(arrowstyle='->', 
                                  color='grey',
                                  alpha=0.5,
                                  lw=1.0))
    
    # Y axis labels
    ax.set_yticks([0, 1, 2, 3])
    ax.set_yticklabels(['Executor', 'Editor', 'Navigator', 'Planner'],
                       fontsize=11, fontweight='bold')
    
    # Horizontal role lines
    for y in [0, 1, 2, 3]:
        ax.axhline(y, color='lightgrey', linewidth=0.8, zorder=0)
    
    ax.set_xlabel('Timestep', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold', color=colour)
    ax.set_facecolor('#f8f9fa')
    
    plt.tight_layout()
    plt.savefig(f'{title.replace(" ", "_")}_sequence.png', 
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

# Plot all four networks
plot_communication_type(plt_fail_traj, 'Matplotlib Fail', 'red')
plot_communication_type(plt_success_traj, 'Matplotlib Success', 'green', max_steps=29)
plot_communication_type(sympy_fail_traj, 'Sympy Fail', 'red')
plot_communication_type(sympy_success_traj, 'Sympy Success', 'green', max_steps=32)

In [ ]:
##FOR PLOTTING FULL LENGTH SUCCESS GRAPHS 
plot_communication_type(plt_success_traj, 'Matplotlib Success', 'green', max_steps=None)
plot_communication_type(sympy_success_traj, 'Sympy Success', 'green', max_steps=None)